# BloodMNIST Centralized CNN Training 

In [ ]:
import os
import time
import random
import subprocess

import numpy as np
import pandas as pd
import psutil
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

import medmnist
from medmnist import INFO
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)


In [ ]:
DATA_ROOT = "./data"
DATASET_NAME = "bloodmnist"
NUM_CLASSES = 8

BATCH_SIZE = 128
EPOCHS = 30
LEARNING_RATE = 0.001
RANDOM_SEED = 42
NUM_WORKERS = 2

MODEL_PATH = "bloodmnist_cnn.pt"
TRAIN_METRICS_PATH = "bloodmnist_training_metrics.csv"
RESOURCE_METRICS_PATH = "bloodmnist_resource_metrics.csv"
FINAL_RESULTS_PATH = "bloodmnist_final_results.csv"


In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 60)
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
print("=" * 60)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(RANDOM_SEED)


In [ ]:
train_transform = transforms.Compose([
    transforms.RandomCrop(size=28, padding=4),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

info = INFO[DATASET_NAME]
DataClass = getattr(medmnist, info["python_class"])

train_dataset = DataClass(split="train", root=DATA_ROOT, transform=train_transform, download=True)
val_dataset = DataClass(split="val", root=DATA_ROOT, transform=val_test_transform, download=True)
test_dataset = DataClass(split="test", root=DATA_ROOT, transform=val_test_transform, download=True)

pin_memory = torch.cuda.is_available()
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=pin_memory)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_memory)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_memory)

print(f"Dataset:            {DATASET_NAME}")
print(f"Training samples:   {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Testing samples:    {len(test_dataset)}")


In [ ]:
# Visualize sample BloodMNIST images with their labels
class_labels = info.get("label", {str(i): f"Class {i}" for i in range(NUM_CLASSES)})

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
fig.suptitle("Sample BloodMNIST Training Images", fontsize=14, fontweight="bold")

for i, ax in enumerate(axes.flat):
    img_tensor, label = train_dataset[i]
    # Denormalize from [-1, 1] to [0, 1]
    img = img_tensor.permute(1, 2, 0).numpy() * 0.5 + 0.5
    img = np.clip(img, 0, 1)
    
    lbl_idx = int(label[0]) if hasattr(label, "__len__") else int(label)
    lbl_name = class_labels.get(str(lbl_idx), f"Class {lbl_idx}")
    
    ax.imshow(img)
    ax.set_title(f"Label {lbl_idx}: {lbl_name}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
class CNN(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        
        # Conv Block 1: 3 x 28 x 28 -> 32 x 28 x 28
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
        )
        
        # Conv Block 2: 32 x 28 x 28 -> 64 x 28 x 28
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )
        
        # Max Pooling: 64 x 28 x 28 -> 64 x 14 x 14
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Global Average Pooling: 64 x 14 x 14 -> 64 x 1 x 1
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Fully Connected Layer: 64 -> num_classes
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.pool(x)
        x = self.global_avg_pool(x)
        x = torch.flatten(x, start_dim=1)
        x = self.fc(x)
        return x

model = CNN(num_classes=NUM_CLASSES).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


In [ ]:
def get_gpu_metrics():
    if not torch.cuda.is_available():
        return 0.0, 0.0
    try:
        result = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader,nounits"],
            encoding="utf-8"
        )
        first_line = result.strip().splitlines()[0]
        values = first_line.split(",")
        return float(values[0].strip()), float(values[1].strip())
    except Exception:
        try:
            allocated_mb = torch.cuda.memory_allocated() / (1024**2)
            return 0.0, float(allocated_mb)
        except Exception:
            return 0.0, 0.0

def get_resource_metrics():
    gpu_util, gpu_mem = get_gpu_metrics()
    return {
        "cpu_percent": psutil.cpu_percent(interval=None),
        "ram_mb": psutil.virtual_memory().used / (1024**2),
        "gpu_percent": gpu_util,
        "gpu_memory_mb": gpu_mem,
    }

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_predictions, all_labels = [], []
    start_time = time.time()

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device, non_blocking=True)
            labels = labels.view(-1).long().to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)

            preds = torch.argmax(outputs, dim=1)
            all_predictions.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return {
        "loss": total_loss / len(dataloader.dataset),
        "accuracy": accuracy_score(all_labels, all_predictions),
        "precision": precision_score(all_labels, all_predictions, average="macro", zero_division=0),
        "recall": recall_score(all_labels, all_predictions, average="macro", zero_division=0),
        "f1": f1_score(all_labels, all_predictions, average="macro", zero_division=0),
        "time": time.time() - start_time,
        "predictions": np.array(all_predictions),
        "labels": np.array(all_labels),
    }


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

training_history = []
resource_history = []
best_val_f1 = -1.0

print("=" * 70)
print(f"STARTING TRAINING ({EPOCHS} EPOCHS)")
print("=" * 70)

total_training_start = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    model.train()
    
    running_loss = 0.0
    all_train_predictions = []
    all_train_labels = []
    
    epoch_cpu = []
    epoch_ram = []
    epoch_gpu = []
    epoch_gpu_memory = []

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.view(-1).long().to(DEVICE, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, dim=1)
        all_train_predictions.extend(preds.detach().cpu().numpy())
        all_train_labels.extend(labels.detach().cpu().numpy())

        resources = get_resource_metrics()
        epoch_cpu.append(resources["cpu_percent"])
        epoch_ram.append(resources["ram_mb"])
        epoch_gpu.append(resources["gpu_percent"])
        epoch_gpu_memory.append(resources["gpu_memory_mb"])

    train_loss = running_loss / len(train_loader.dataset)
    train_accuracy = accuracy_score(all_train_labels, all_train_predictions)
    train_precision = precision_score(all_train_labels, all_train_predictions, average="macro", zero_division=0)
    train_recall = recall_score(all_train_labels, all_train_predictions, average="macro", zero_division=0)
    train_f1 = f1_score(all_train_labels, all_train_predictions, average="macro", zero_division=0)

    val_metrics = evaluate(model, val_loader, criterion, DEVICE)
    epoch_time = time.time() - epoch_start

    avg_cpu = float(np.mean(epoch_cpu))
    max_cpu = float(np.max(epoch_cpu))
    avg_ram = float(np.mean(epoch_ram))
    max_ram = float(np.max(epoch_ram))
    avg_gpu = float(np.mean(epoch_gpu))
    max_gpu = float(np.max(epoch_gpu))
    avg_gpu_memory = float(np.mean(epoch_gpu_memory))
    max_gpu_memory = float(np.max(epoch_gpu_memory))

    training_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "train_precision": train_precision,
        "train_recall": train_recall,
        "train_f1": train_f1,
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_f1": val_metrics["f1"],
        "epoch_time_seconds": epoch_time,
    })

    resource_history.append({
        "epoch": epoch,
        "avg_cpu_percent": avg_cpu,
        "max_cpu_percent": max_cpu,
        "avg_ram_mb": avg_ram,
        "max_ram_mb": max_ram,
        "avg_gpu_percent": avg_gpu,
        "max_gpu_percent": max_gpu,
        "avg_gpu_memory_mb": avg_gpu_memory,
        "max_gpu_memory_mb": max_gpu_memory,
        "epoch_time_seconds": epoch_time,
    })

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        torch.save(model.state_dict(), MODEL_PATH)
        best_marker = " <-- BEST"
    else:
        best_marker = ""

    print(f"Epoch [{epoch:02d}/{EPOCHS}] | Time: {epoch_time:.2f}s")
    print(f"  Train | Loss: {train_loss:.4f} | Acc: {train_accuracy:.4f} | F1: {train_f1:.4f}")
    print(f"  Val   | Loss: {val_metrics['loss']:.4f} | Acc: {val_metrics['accuracy']:.4f} | F1: {val_metrics['f1']:.4f}{best_marker}")
    print(f"  Res   | CPU: {avg_cpu:.1f}% | RAM: {avg_ram:.1f}MB | GPU: {avg_gpu:.1f}% | GPU Mem: {avg_gpu_memory:.1f}MB")

total_training_time = time.time() - total_training_start
print("\n" + "=" * 70)
print(f"TRAINING COMPLETE in {total_training_time:.2f}s | Best Val F1: {best_val_f1:.4f}")
print("=" * 70)


In [ ]:
train_df = pd.DataFrame(training_history)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(train_df["epoch"], train_df["train_loss"], label="Train Loss", color="royalblue", marker="o")
axes[0].plot(train_df["epoch"], train_df["val_loss"], label="Val Loss", color="crimson", marker="s")
axes[0].set_title("Training & Validation Loss", fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("CrossEntropy Loss")
axes[0].grid(True, linestyle="--", alpha=0.6)
axes[0].legend()

# Accuracy
axes[1].plot(train_df["epoch"], train_df["train_accuracy"], label="Train Acc", color="royalblue", marker="o")
axes[1].plot(train_df["epoch"], train_df["val_accuracy"], label="Val Acc", color="forestgreen", marker="s")
axes[1].set_title("Training & Validation Accuracy", fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].grid(True, linestyle="--", alpha=0.6)
axes[1].legend()

# F1-Score
axes[2].plot(train_df["epoch"], train_df["train_f1"], label="Train Macro F1", color="royalblue", marker="o")
axes[2].plot(train_df["epoch"], train_df["val_f1"], label="Val Macro F1", color="darkorange", marker="s")
axes[2].set_title("Training & Validation Macro F1", fontweight="bold")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("F1 Score")
axes[2].grid(True, linestyle="--", alpha=0.6)
axes[2].legend()

plt.tight_layout()
plt.show()


In [ ]:
res_df = pd.DataFrame(resource_history)

fig, axes = plt.subplots(2, 2, figsize=(15, 9))

# CPU Usage
axes[0, 0].plot(res_df["epoch"], res_df["avg_cpu_percent"], label="Avg CPU %", color="darkcyan", marker="o")
axes[0, 0].plot(res_df["epoch"], res_df["max_cpu_percent"], label="Max CPU %", color="steelblue", linestyle="--")
axes[0, 0].set_title("CPU Utilization (%)", fontweight="bold")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("CPU %")
axes[0, 0].grid(True, linestyle="--", alpha=0.6)
axes[0, 0].legend()

# RAM Usage
axes[0, 1].plot(res_df["epoch"], res_df["avg_ram_mb"], label="Avg RAM (MB)", color="purple", marker="o")
axes[0, 1].plot(res_df["epoch"], res_df["max_ram_mb"], label="Max RAM (MB)", color="orchid", linestyle="--")
axes[0, 1].set_title("System RAM Usage (MB)", fontweight="bold")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("RAM (MB)")
axes[0, 1].grid(True, linestyle="--", alpha=0.6)
axes[0, 1].legend()

# GPU Usage
axes[1, 0].plot(res_df["epoch"], res_df["avg_gpu_percent"], label="Avg GPU %", color="forestgreen", marker="o")
axes[1, 0].plot(res_df["epoch"], res_df["max_gpu_percent"], label="Max GPU %", color="limegreen", linestyle="--")
axes[1, 0].set_title("GPU Utilization (%)", fontweight="bold")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("GPU %")
axes[1, 0].grid(True, linestyle="--", alpha=0.6)
axes[1, 0].legend()

# GPU Memory
axes[1, 1].plot(res_df["epoch"], res_df["avg_gpu_memory_mb"], label="Avg GPU Mem (MB)", color="chocolate", marker="o")
axes[1, 1].plot(res_df["epoch"], res_df["max_gpu_memory_mb"], label="Max GPU Mem (MB)", color="sandybrown", linestyle="--")
axes[1, 1].set_title("GPU Memory Usage (MB)", fontweight="bold")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Memory (MB)")
axes[1, 1].grid(True, linestyle="--", alpha=0.6)
axes[1, 1].legend()

plt.tight_layout()
plt.show()


In [ ]:
print("Loading best model weights from:", MODEL_PATH)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))

test_start = time.time()
test_metrics = evaluate(model, test_loader, criterion, DEVICE)
test_time = time.time() - test_start

print("=" * 50)
print("FINAL TEST EVALUATION")
print("=" * 50)
print(f"Test Loss:      {test_metrics['loss']:.4f}")
print(f"Test Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"Test Precision: {test_metrics['precision']:.4f}")
print(f"Test Recall:    {test_metrics['recall']:.4f}")
print(f"Test F1-Score:  {test_metrics['f1']:.4f}")
print(f"Test Time:      {test_time:.2f} seconds")
print("=" * 50)


In [ ]:
cm = confusion_matrix(test_metrics["labels"], test_metrics["predictions"])
class_names = [class_labels.get(str(i), f"C{i}") for i in range(NUM_CLASSES)]

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)

ax.set(
    xticks=np.arange(cm.shape[1]),
    yticks=np.arange(cm.shape[0]),
    xticklabels=class_names,
    yticklabels=class_names,
    title="BloodMNIST Confusion Matrix",
    ylabel="True label",
    xlabel="Predicted label",
)

plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# Text annotations in each cell
thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], "d"),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")

plt.tight_layout()
plt.show()

# Classification Report
print("\nClassification Report:\n")
print(classification_report(test_metrics["labels"], test_metrics["predictions"], target_names=class_names, zero_division=0))


In [ ]:
all_resource_data = pd.DataFrame(resource_history)
overall_resource_metrics = {
    "avg_cpu_percent": float(all_resource_data["avg_cpu_percent"].mean()),
    "max_cpu_percent": float(all_resource_data["max_cpu_percent"].max()),
    "avg_ram_mb": float(all_resource_data["avg_ram_mb"].mean()),
    "max_ram_mb": float(all_resource_data["max_ram_mb"].max()),
    "avg_gpu_percent": float(all_resource_data["avg_gpu_percent"].mean()),
    "max_gpu_percent": float(all_resource_data["max_gpu_percent"].max()),
    "avg_gpu_memory_mb": float(all_resource_data["avg_gpu_memory_mb"].mean()),
    "max_gpu_memory_mb": float(all_resource_data["max_gpu_memory_mb"].max()),
    "total_training_time_seconds": total_training_time,
}

final_results = {
    "model": "Centralized CNN",
    "dataset": DATASET_NAME,
    "train_samples": len(train_dataset),
    "val_samples": len(val_dataset),
    "test_samples": len(test_dataset),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "total_parameters": total_params,
    "trainable_parameters": trainable_params,
    "test_loss": test_metrics["loss"],
    "test_accuracy": test_metrics["accuracy"],
    "test_precision": test_metrics["precision"],
    "test_recall": test_metrics["recall"],
    "test_f1": test_metrics["f1"],
    "test_time_seconds": test_time,
    "total_training_time_seconds": total_training_time,
    **overall_resource_metrics,
}

train_df.to_csv(TRAIN_METRICS_PATH, index=False)
res_df.to_csv(RESOURCE_METRICS_PATH, index=False)
pd.DataFrame([final_results]).to_csv(FINAL_RESULTS_PATH, index=False)

print("Saved files:")
print(f"1. {MODEL_PATH}")
print(f"2. {TRAIN_METRICS_PATH}")
print(f"3. {RESOURCE_METRICS_PATH}")
print(f"4. {FINAL_RESULTS_PATH}")
